# Connect to St. Louis Federal Reserve Economic Data (FRED) API and retrieve data
This notebook will be used as an introductory notebook to start organizing the data for the St. Louis Federal reserve. This API will be used predominantly for the macroeconomic data that we will be pulling. It will likely have different time points, so I think I will need to pull the series indvidually to determine which can be concatenated together, we will then have to decide how we deal with heterogenous time series data.

*Note* - A free api key is required to access the data from the St. Louis Federal Reserve. If you need to generate an API key visit: https://fred.stlouisfed.org/docs/api/api_key.html

## Libraries

In [10]:
import numpy as np
import pandas as pd
import altair as alt

from fredapi import Fred

# Disable the max rows limit in Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

Now, that we have imported our libraries we need to connect to the fred_api. This requires our unique api_key and instantiating a fred instance using Fred. 

In [11]:
# Need an API key to access the FRED data
api_key = input("Enter your FRED API key: ")
fred = Fred(api_key=api_key)
print("Successfully connected to FRED API!")

Successfully connected to FRED API!


Alright, now let's start to pull data from the api. The hardest part of this is figuring out what the series name is that we want to pull. Let's start with something pretty straight forward like the Nominal GDP and Real GDP. This will give us a chance to see how far back we can obtain data as well. The nomial GDP should just be 'GDP' this is the unadjusted dollar value of Gross Domestic Product the real GDP should be under 'GDPC1', this is the inflation adjusted GDP in absolute US dollars.

In [12]:
# Call the fred api for GDP data
real_gdp = fred.get_series("GDPC1").dropna()
nom_gdp = fred.get_series("GDP").dropna()
print(f"We have {len(nom_gdp)} nominal GDP data points and {len(real_gdp)} real GDP data points.")

We have 316 nominal GDP data points and 316 real GDP data points.


Alright, we have 316 data points, considering these values are usually published quarterly this should be a significant amount of time so let's take a look at how far back this goes. 

In [13]:
nom_gdp.head()

1947-01-01    243.164
1947-04-01    245.968
1947-07-01    249.585
1947-10-01    259.745
1948-01-01    265.742
dtype: float64

Dating all the way back to 1947. Excellent. It does appear to be quarterly. Let's see if the most recent data lines up with what is published to verify the data and see whether or not the date is for the date the data was 'published' or more likely the start or end of the period the GDP is related to.

In [14]:
nom_gdp.tail()

2024-10-01    29825.182
2025-01-01    30042.113
2025-04-01    30485.729
2025-07-01    31098.027
2025-10-01    31490.070
dtype: float64

Okay, so the value corresponds to the BEGINNING of the period that the GDP is reported for. This is going to be a really important distinction to avoid data leakage. We wont have the values at the beginning of the quarter, likely we wont even have them until a quarter after. I'm pretty sure the Q4 2025 GDP numbers were just eleased today (2/2/26). Another thing that we need to think about is how these values are typically used. it's very rare to have absolute values used, so we may want to calculate both Quarter over Quarter (QoQ) changes and Year over Year changes (YoY).

In [15]:
# First we need to convert to a dataframe
real_gdp_df = real_gdp.reset_index()
real_gdp_df.columns = ["date", "real_gdp"]
# Calculate the quarter-over-quarter and year-over-year percentage changes
real_gdp_df['QoQ_%'] = round(real_gdp_df['real_gdp'].pct_change() * 100,2)
real_gdp_df['YoY_%'] = round(real_gdp_df['real_gdp'].pct_change(periods=4) * 100,2)
# Drop the NA values we created
real_gdp_df.dropna(inplace=True)
# Let's reset the index  because we may merge on this later
real_gdp_df.set_index("date", inplace=True)
# Let's take a look at the data to see if its accurate
real_gdp_df.tail()

,real_gdp,QoQ_%,YoY_%
date,,,
2024-10-01,23586.542,0.46,2.40
2025-01-01,23548.210,-0.16,2.02
2025-04-01,23770.976,0.95,2.08
2025-07-01,24026.834,1.08,2.34
2025-10-01,24111.830,0.35,2.23


Just for interest sake, let's create two plots of data to take a look at how it's adjusted voer time.

In [16]:
df = real_gdp_df.reset_index()[['date', 'YoY_%']].copy()
df.rename(columns={'YoY_%': 'value'}, inplace=True)
# Create above/below zero columns
df['above'] = df['value'].clip(lower=0)
df['below'] = df['value'].clip(upper=0)

# Base x-axis encoding
base = alt.Chart(df).encode(x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)))

# Green area above zero
area_above = base.mark_area(
    color='darkgreen',
    opacity=0.3
).encode(
    y=alt.Y('above:Q', axis = alt.Axis(grid=False)),
    y2=alt.Y2(datum=0)
)

# Red area below zero
area_below = base.mark_area(
    color='red',
    opacity=0.3
).encode(
    y=alt.Y('below:Q', axis = alt.Axis(grid=False, title='YoY % Change')),
    y2=alt.Y2(datum=0)
)

# The main line to show the real GDP YoY percentage change
line = base.mark_line(
    color='black',
    strokeWidth=2
).encode(
    y=alt.Y('value:Q', axis = alt.Axis(grid=False))
)

# Zero rule to see what no change would be
zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
    color='maroon',
    strokeWidth=1,
    strokeDash=[4, 4]
).encode(
    y=alt.Y('y:Q', axis = alt.Axis(grid=False))
)

# Puting it all together
chart = (area_above + area_below + line + zero_line).properties(
    width=900,
    height=600,
    title='Real GDP Year over Year Percentage Change'
)
# printing to screen
chart

alt.LayerChart(...)

*Note* - This figure brings up a very intersting issue. When looking at the COVID-19 pandemic, we can see one of the largest drops in economic output in US history, followed by one of the largest. However, the recovery may be misleading as it is based on the YoY figure (which was horrific during covid). Just something to think about.

Alright, this has been a great proof of concept ut now let's think about all of the different data that we may or may not want to use and write down their series name for ease.

### GDP and Debt
Nominal GDP = fred.get_series("GDP")  
Real GDP = fred.get_series("GDPC1")  
US Debt to GDP = fred.get_series("GFDEGDQ1885")  
Interest Payment on debt = fred.get_series("A091RC1Q027SBEA")  

### Inflation Data  
Consumer Price Index (CPI): fred.get_series("CPIAUCSL")  
Core PCI: fred.get_series("CPILFESL")  
Personal Consumption Expenditure (PCE): fred.get_series("PCEPI")  
Core PCE: fred.get_series("PCEPILFE")  
Producer Price Index (PPI): fred.get_series("PPIFIS")  

### Employment Data  
Unemployment Rate: fred.get_series("UNRATE")  
Initial Jobless Claims: fred.get_series("ICSA")  
Continued Jobless claims: fred.get_series("CCSA")  
Teenager unemployment rate: fred.get_series("LNS14000012")  
Adult unemployment rate: fred.get_series("LNS14000025")  
Male Unemployment rate: fred.get_series("LNS14000001")  
Female Unemployment rate: fred.get_series("LNS14000002")  
Average Duration of Unemployment: fred.get_series("UEMPMEAN")  

### Treasury Yield Data (cost of lending)  
1-Month Yield: fred.get_series("DGS1MO")  
3-Month Yield: fred.get_series("DGS3MO")  
6-Month Yield: fred.get_series("DGS6MO")  
1-Year Yield: fred.get_series("DGS1")  
2-Year Yield: fred.get_series("DGS2")  
5-Year Yield: fred.get_series("DGS5")  
10-Year Yield: fred.get_series("DGS10")  
30-Year Yield: fred.get_series("DGS30")  

These are the main one's I can think of for now but we can add to it as we see fit.

## Full Data Pull
Alright, now that we have it all figured out let's create a function to pull all of the data. From here we will be able to look at the frequency and line everything up together in a dataframe.

In [17]:
def get_fred_data():
    """Fetch the latest data from FRED."""
    try:
        # Get the latest data from fredapi
        fred_data = {
            "nominal_GDP" : fred.get_series("GDP"),  
            "real_GDP" : fred.get_series("GDPC1"),
            "debt_to_GDP" : fred.get_series("GFDEGDQ188S"),
            "debt_interest" : fred.get_series("A091RC1Q027SBEA"),
            "consumer_price_index" : fred.get_series("CPIAUCSL"),
            "core_PCI" : fred.get_series("CPILFESL"),
            "personal_consumption_expenditure" : fred.get_series("PCEPI"),
            "core_PCE" : fred.get_series("PCEPILFE"),
            "producer_price_index" : fred.get_series("PPIFIS"),
            "unemployment_rate" : fred.get_series("UNRATE"),
            "initial_jobless_claims" : fred.get_series("ICSA"),
            "continued_jobless_claims" : fred.get_series("CCSA"),
            "teenager_unemployment_rate" : fred.get_series("LNS14000012"),
            "adult_unemployment_rate" : fred.get_series("LNS14000025"),
            "male_unemployment_rate" : fred.get_series("LNS14000001"),
            "female_unemployment_rate" : fred.get_series("LNS14000002"),
            "average_duration_of_unemployment" : fred.get_series("UEMPMEAN"),
            "one_month_yield" : fred.get_series("DGS1MO"),
            "three_month_yield" : fred.get_series("DGS3MO"),
            "six_month_yield" : fred.get_series("DGS6MO"),
            "one_year_yield" : fred.get_series("DGS1"),
            "two_year_yield" : fred.get_series("DGS2"),
            "five_year_yield" : fred.get_series("DGS5"),
            "ten_year_yield" : fred.get_series("DGS10"),
            "thirty_year_yield" : fred.get_series("DGS30"),
        }

        if not fred_data["unemployment_rate"].empty:

            return {
                "fred_data": fred_data,
                "source": "FRED",
            }

    except Exception as e:
        print(f"Unemployment Rate FRED failed: {e}")

Alright, now lets run it and see if we get the data we want.

In [18]:
fred_data = get_fred_data()
fred_data

{'fred_data': {'nominal_GDP': 1946-01-01          NaN
  1946-04-01          NaN
  1946-07-01          NaN
  1946-10-01          NaN
  1947-01-01      243.164
                  ...    
  2024-10-01    29825.182
  2025-01-01    30042.113
  2025-04-01    30485.729
  2025-07-01    31098.027
  2025-10-01    31490.070
  Length: 320, dtype: float64,
  'real_GDP': 1947-01-01     2182.681
  1947-04-01     2176.892
  1947-07-01     2172.432
  1947-10-01     2206.452
  1948-01-01     2239.682
                  ...    
  2024-10-01    23586.542
  2025-01-01    23548.210
  2025-04-01    23770.976
  2025-07-01    24026.834
  2025-10-01    24111.830
  Length: 316, dtype: float64,
  'debt_to_GDP': 1966-01-01     40.33999
  1966-04-01     39.26763
  1966-07-01     39.62091
  1966-10-01     39.51977
  1967-01-01     39.20383
                  ...    
  2024-07-01    120.17172
  2024-10-01    121.43633
  2025-01-01    120.54515
  2025-04-01    118.78171
  2025-07-01    121.02875
  Length: 239, dtype: flo

Excellent, so now we should be able to simply merge all of these series together based on their index.

In [19]:
fred_df = pd.DataFrame({name: series for name, series in fred_data["fred_data"].items()})
print(f"We have {len(fred_df)} data points in the FRED dataframe.")
print(fred_df.head())

We have 20125 data points in the FRED dataframe.
            nominal_GDP  real_GDP  debt_to_GDP  debt_interest  \
1946-01-01          NaN       NaN          NaN            NaN   
1946-04-01          NaN       NaN          NaN            NaN   
1946-07-01          NaN       NaN          NaN            NaN   
1946-10-01          NaN       NaN          NaN            NaN   
1947-01-01      243.164  2182.681          NaN          5.352   

            consumer_price_index  core_PCI  personal_consumption_expenditure  \
1946-01-01                   NaN       NaN                               NaN   
1946-04-01                   NaN       NaN                               NaN   
1946-07-01                   NaN       NaN                               NaN   
1946-10-01                   NaN       NaN                               NaN   
1947-01-01                 21.48       NaN                               NaN   

            core_PCE  producer_price_index  unemployment_rate  ...  \
1946-01-0

Alright, this looks great, lets now export it to a csv

In [20]:
fred_df.to_csv('fred_data.csv', index_label='date')